In [1]:
from pathlib import Path
from typing import Optional
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, save_npz
from scipy.stats import spearmanr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Lasso, LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)

In [7]:
# ============================================================
# PATHS
# ============================================================

THESIS_DIR = Path(
    "/Users/sonalprasad/Documents/Thesis"
)

INPUT_CSV_PATH = (
    THESIS_DIR
    / "DAICWOZ_Initial.csv"
)

TFIDF_OUTPUT_DIR = (
    THESIS_DIR
    / "TFIDF/V2"
)

TFIDF_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DATASET COLUMNS
# ============================================================

PARTICIPANT_ID_COLUMN = "Participant_ID"
TRANSCRIPT_PATH_COLUMN = "Transcript_Path"
DATASET_COLUMN = "Dataset"

BINARY_TARGET_COLUMN = "PHQ8_Binary"
REGRESSION_TARGET_COLUMN = "PHQ8_Score"

EXPECTED_DATASET_VALUES = {
    "train",
    "dev",
    "test",
}


# ============================================================
# TRANSCRIPT COLUMNS
# ============================================================

TRANSCRIPT_SPEAKER_COLUMN = "speaker"
TRANSCRIPT_RESPONSE_COLUMN = "response"

PARTICIPANT_SPEAKER_VALUE = "Participant"


# ============================================================
# TEXT PREPROCESSING
# ============================================================

KEEP_ANNOTATIONS = True

ANNOTATION_PREFIX = "annotation_"

PRESERVE_APOSTROPHES = True


# ============================================================
# TF-IDF DEFAULTS
# ============================================================

TFIDF_STOP_WORDS = None

TFIDF_NORM = "l2"

USE_IDF = True

SUBLINEAR_TF = True

MIN_DF = 2

MAX_DF = 0.9


# ============================================================
# PHQ-8 LASSO SEARCH
# ============================================================

REGRESSION_ALPHAS = (
    0.0001,
    0.0005,
    0.001,
    0.0025,
    0.005,
    0.01,
    0.025,
    0.05,
    0.1,
    0.25,
    0.5,
)

LASSO_MAX_ITER = 20_000

LASSO_TOLERANCE = 1e-4


# ============================================================
# BINARY L1 LOGISTIC REGRESSION SEARCH
# ============================================================

CLASSIFICATION_C_VALUES = (
    0.001,
    0.0025,
    0.005,
    0.01,
    0.025,
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
    2.5,
    5.0,
    10.0,
)

LOGISTIC_MAX_ITER = 10_000

LOGISTIC_CLASS_WEIGHT = "balanced"

RANDOM_STATE = 42


# ============================================================
# OPTIONAL FEATURE COUNT CONSTRAINTS
# ============================================================

# Keep both None if you want selection based only on
# validation performance.

MIN_SELECTED_FEATURES: Optional[int] = None

MAX_SELECTED_FEATURES: Optional[int] = None


# ============================================================
# TOKENIZATION
# ============================================================

# Preserves:
# don't
# can't
# annotation_sigh
# annotation_makes_sound_with_mouth

TOKEN_PATTERN = (
    r"(?u)\b[a-zA-Z0-9_]+(?:'[a-zA-Z0-9_]+)?\b"
)

In [2]:
# # ============================================================
# # PATHS
# # ============================================================

# THESIS_DIR = Path(
#     "/Users/sonalprasad/Documents/Thesis"
# )

# INPUT_CSV_PATH = (
#     THESIS_DIR
#     / "DAICWOZ_Initial.csv"
# )

# TFIDF_OUTPUT_DIR = (
#     THESIS_DIR
#     / "TFIDF"
# )

# TFIDF_OUTPUT_DIR.mkdir(
#     parents=True,
#     exist_ok=True,
# )


# # ============================================================
# # MAIN DATASET COLUMNS
# # ============================================================

# PARTICIPANT_ID_COLUMN = "Participant_ID"
# TRANSCRIPT_PATH_COLUMN = "Transcript_Path"
# DATASET_COLUMN = "Dataset"

# BINARY_TARGET_COLUMN = "PHQ8_Binary"
# REGRESSION_TARGET_COLUMN = "PHQ8_Score"


# # ============================================================
# # DAIC-WOZ TRANSCRIPT FORMAT
# # ============================================================

# TRANSCRIPT_COLUMNS = [
#     "start_time",
#     "end_time",
#     "speaker",
#     "text",
# ]

# PARTICIPANT_SPEAKER_VALUE = "Participant"


# # ============================================================
# # TEXT PREPROCESSING
# # ============================================================

# KEEP_ANNOTATIONS = True

# ANNOTATION_PREFIX = "annotation_"

# PRESERVE_APOSTROPHES = True


# # ============================================================
# # TF-IDF SETTINGS
# # ============================================================

# TFIDF_STOP_WORDS = None

# TFIDF_NORM = "l2"

# USE_IDF = True

# SUBLINEAR_TF = True

# # A term must appear in at least this many TRAIN participants.
# MIN_DF = 2

# # 1.0 means do not remove terms based on high document frequency.
# MAX_DF = 1.0


# # ============================================================
# # PHQ-8 LASSO SETTINGS
# # ============================================================

# REGRESSION_ALPHAS = (
#     0.0001,
#     0.0005,
#     0.001,
#     0.0025,
#     0.005,
#     0.01,
#     0.025,
#     0.05,
#     0.1,
#     0.25,
#     0.5,
# )

# LASSO_MAX_ITER = 20_000

# LASSO_TOLERANCE = 1e-4


# # ============================================================
# # BINARY L1 LOGISTIC REGRESSION SETTINGS
# # ============================================================

# CLASSIFICATION_C_VALUES = (
#     0.001,
#     0.0025,
#     0.005,
#     0.01,
#     0.025,
#     0.05,
#     0.1,
#     0.25,
#     0.5,
#     1.0,
#     2.5,
#     5.0,
#     10.0,
# )

# LOGISTIC_MAX_ITER = 10_000

# LOGISTIC_CLASS_WEIGHT = "balanced"

# RANDOM_STATE = 42


# # ============================================================
# # OPTIONAL SELECTED-FEATURE CONSTRAINTS
# # ============================================================

# # Leave these as None if you want the best validation model,
# # regardless of how many features survive.

# MIN_SELECTED_FEATURES: Optional[int] = None

# MAX_SELECTED_FEATURES: Optional[int] = None


# # ============================================================
# # TOKENIZATION
# # ============================================================

# # Preserves:
# # don't
# # can't
# # annotation_sigh
# # annotation_makes_sound_with_mouth

# TOKEN_PATTERN = (
#     r"(?u)\b[a-zA-Z0-9_]+(?:'[a-zA-Z0-9_]+)?\b"
# )

In [76]:
# Dataset helper functions

def load_initial_dataset(
    csv_path: Path,
) -> pd.DataFrame:

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found:\n{csv_path}"
        )

    return pd.read_csv(csv_path)


def validate_initial_dataset(
    df: pd.DataFrame,
) -> None:

    required_columns = [
        PARTICIPANT_ID_COLUMN,
        "Folder",
        "Audio_Filename",
        "Output_Audio_Path",
        TRANSCRIPT_PATH_COLUMN,
        DATASET_COLUMN,
        "Gender",
        BINARY_TARGET_COLUMN,
        REGRESSION_TARGET_COLUMN,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns:\n{missing_columns}"
        )

    # One row per participant
    duplicated_participants = (
        df.loc[
            df[PARTICIPANT_ID_COLUMN].duplicated(
                keep=False
            ),
            PARTICIPANT_ID_COLUMN,
        ]
        .tolist()
    )

    if duplicated_participants:
        raise ValueError(
            "Participant_ID must be unique.\n"
            f"Duplicates:\n"
            f"{sorted(set(duplicated_participants))}"
        )

    # Normalize split names
    df[DATASET_COLUMN] = (
        df[DATASET_COLUMN]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    expected_splits = {
        "train",
        "dev",
        "test",
    }

    unexpected_splits = (
        set(
            df[DATASET_COLUMN].unique()
        )
        - expected_splits
    )

    if unexpected_splits:
        raise ValueError(
            f"Unexpected Dataset values:\n"
            f"{unexpected_splits}"
        )

    # Numeric targets
    df[BINARY_TARGET_COLUMN] = pd.to_numeric(
        df[BINARY_TARGET_COLUMN],
        errors="coerce",
    )

    df[REGRESSION_TARGET_COLUMN] = pd.to_numeric(
        df[REGRESSION_TARGET_COLUMN],
        errors="coerce",
    )

    invalid_binary_values = (
        set(
            df[BINARY_TARGET_COLUMN]
            .dropna()
            .unique()
        )
        - {0, 1}
    )

    if invalid_binary_values:
        raise ValueError(
            "PHQ8_Binary must contain only 0 and 1.\n"
            f"Found:\n{invalid_binary_values}"
        )

In [77]:
# Transcript reader
def read_daicwoz_transcript(
    transcript_path: str | Path,
) -> pd.DataFrame:

    transcript_path = Path(
        transcript_path
    )

    if not transcript_path.exists():
        raise FileNotFoundError(
            f"Transcript not found:\n"
            f"{transcript_path}"
        )

    transcript_df = pd.read_csv(
        transcript_path,
        sep="\t",
        header=None,
        names=TRANSCRIPT_COLUMNS,
        dtype=str,
        keep_default_na=False,
    )

    return transcript_df

In [78]:
ANNOTATION_PATTERN = re.compile(
    r"<([^<>]+)>|\[([^\[\]]+)\]"
)

def annotation_to_token(
    annotation_text: str,
) -> str:

    annotation_text = (
        annotation_text
        .strip()
        .lower()
    )

    # Remove apostrophes inside annotations.
    annotation_text = re.sub(
        r"[’']",
        "",
        annotation_text,
    )

    # Convert punctuation/spaces to underscores.
    annotation_text = re.sub(
        r"[^a-z0-9]+",
        "_",
        annotation_text,
    )

    annotation_text = (
        annotation_text
        .strip("_")
    )

    annotation_text = re.sub(
        r"_+",
        "_",
        annotation_text,
    )

    if not annotation_text:
        return ""

    return (
        f"{ANNOTATION_PREFIX}"
        f"{annotation_text}"
    )


def normalize_annotations(
    text: str,
) -> str:

    if not KEEP_ANNOTATIONS:
        return ANNOTATION_PATTERN.sub(
            " ",
            text,
        )

    def replace_annotation(
        match: re.Match,
    ) -> str:

        # One capture group corresponds to <>
        # the other corresponds to [].
        annotation_text = (
            match.group(1)
            if match.group(1) is not None
            else match.group(2)
        )

        annotation_token = (
            annotation_to_token(
                annotation_text
            )
        )

        if not annotation_token:
            return " "

        return f" {annotation_token} "

    return ANNOTATION_PATTERN.sub(
        replace_annotation,
        text,
    )

def extract_annotation_tokens(
    text: str,
) -> list[str]:

    tokens = []

    for match in ANNOTATION_PATTERN.finditer(text):

        annotation_text = (
            match.group(1)
            if match.group(1) is not None
            else match.group(2)
        )

        token = annotation_to_token(
            annotation_text
        )

        if token:
            tokens.append(token)

    return tokens

In [79]:
# Text processing
def normalize_apostrophes(
    text: str,
) -> str:

    return (
        text
        .replace("’", "'")
        .replace("‘", "'")
    )


def remove_punctuation(
    text: str,
) -> str:

    if PRESERVE_APOSTROPHES:
        allowed_extra = "_'"
    else:
        allowed_extra = "_"

    return re.sub(
        rf"[^a-zA-Z0-9\s{re.escape(allowed_extra)}]",
        " ",
        text,
    )


def normalize_whitespace(
    text: str,
) -> str:

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def preprocess_text(
    text: str,
) -> str:

    if pd.isna(text):
        return ""

    text = str(text)

    # Important:
    # annotations must be converted before punctuation removal.
    text = normalize_annotations(
        text
    )

    text = normalize_apostrophes(
        text
    )

    text = text.lower()

    text = remove_punctuation(
        text
    )

    text = normalize_whitespace(
        text
    )

    return text

In [80]:
# Extract participant-only text
def extract_participant_text(
    transcript_df: pd.DataFrame,
) -> tuple[str, int]:

    participant_rows = (
        transcript_df.loc[
            transcript_df["speaker"]
            .astype(str)
            .str.strip()
            .str.lower()
            == PARTICIPANT_SPEAKER_VALUE.lower()
        ]
    )

    participant_responses = (
        participant_rows["text"]
        .astype(str)
        .tolist()
    )

    participant_text = " ".join(
        participant_responses
    )

    return (
        participant_text,
        len(participant_responses),
    )


def count_annotations(
    text: str,
) -> int:

    return len(
        ANNOTATION_PATTERN.findall(
            text
        )
    )

In [81]:
# Build one document per participant
def build_participant_documents(
    df: pd.DataFrame,
) -> pd.DataFrame:

    records = []

    failures = []

    total = len(df)

    for counter, (_, row) in enumerate(
        df.iterrows(),
        start=1,
    ):

        participant_id = row[
            PARTICIPANT_ID_COLUMN
        ]

        transcript_path = row[
            TRANSCRIPT_PATH_COLUMN
        ]

        print(
            f"[{counter}/{total}] "
            f"Participant {participant_id}"
        )

        try:

            transcript_df = (
                read_daicwoz_transcript(
                    transcript_path
                )
            )

            (
                raw_participant_text,
                utterance_count,
            ) = extract_participant_text(
                transcript_df
            )

            annotation_tokens = extract_annotation_tokens(
                raw_participant_text
            )

            cleaned_participant_text = (
                preprocess_text(
                    raw_participant_text
                )
            )

            records.append(
                {
                    PARTICIPANT_ID_COLUMN:
                        participant_id,

                    DATASET_COLUMN:
                        row[DATASET_COLUMN],

                    TRANSCRIPT_PATH_COLUMN:
                        transcript_path,

                    BINARY_TARGET_COLUMN:
                        row[
                            BINARY_TARGET_COLUMN
                        ],

                    REGRESSION_TARGET_COLUMN:
                        row[
                            REGRESSION_TARGET_COLUMN
                        ],

                    "participant_utterance_count":
                        utterance_count,

                    "annotation_count":
                        len(annotation_tokens),

                    "annotation_tokens":
                        "; ".join(annotation_tokens),

                    "raw_participant_text":
                        raw_participant_text,

                    "cleaned_participant_text":
                        cleaned_participant_text,
                }
            )

        except Exception as exc:

            failures.append(
                {
                    PARTICIPANT_ID_COLUMN:
                        participant_id,

                    TRANSCRIPT_PATH_COLUMN:
                        transcript_path,

                    "error":
                        str(exc),
                }
            )

    documents_df = pd.DataFrame(
        records
    )

    if failures:

        failures_df = pd.DataFrame(
            failures
        )

        failure_path = (
            TFIDF_OUTPUT_DIR
            / "transcript_processing_failures.csv"
        )

        failures_df.to_csv(
            failure_path,
            index=False,
        )

        print(
            f"\nFailed transcripts: "
            f"{len(failures)}"
        )

        print(
            f"Failures saved to:\n"
            f"{failure_path}"
        )

    return documents_df

In [82]:
# TF-IDF helper functions
def build_tfidf_vectorizer(
    ngram_range: tuple[int, int],
    max_features: Optional[int],
) -> TfidfVectorizer:

    return TfidfVectorizer(
        lowercase=False,
        token_pattern=TOKEN_PATTERN,
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=MIN_DF,
        max_df=MAX_DF,
        stop_words=TFIDF_STOP_WORDS,
        norm=TFIDF_NORM,
        use_idf=USE_IDF,
        sublinear_tf=SUBLINEAR_TF,
    )


def fit_tfidf(
    train_df: pd.DataFrame,
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    ngram_range: tuple[int, int],
    max_features: Optional[int],
):

    vectorizer = build_tfidf_vectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
    )

    # Fit ONLY on train.
    X_train = vectorizer.fit_transform(
        train_df[
            "cleaned_participant_text"
        ].fillna("")
    )

    X_dev = vectorizer.transform(
        dev_df[
            "cleaned_participant_text"
        ].fillna("")
    )

    X_test = vectorizer.transform(
        test_df[
            "cleaned_participant_text"
        ].fillna("")
    )

    return (
        vectorizer,
        X_train.tocsr(),
        X_dev.tocsr(),
        X_test.tocsr(),
    )

In [83]:
# Vocabulary metadata helpers
def classify_feature_type(
    feature: str,
) -> str:

    tokens = feature.split()

    contains_annotation = any(
        token.startswith(
            ANNOTATION_PREFIX
        )
        for token in tokens
    )

    if len(tokens) == 1:

        if contains_annotation:
            return "unigram_annotation"

        return "unigram_word"

    if contains_annotation:
        return "bigram_with_annotation"

    return "bigram_word_word"


def build_vocabulary_dataframe(
    vectorizer: TfidfVectorizer,
) -> pd.DataFrame:

    feature_names = (
        vectorizer
        .get_feature_names_out()
    )

    records = []

    for feature_index, (
        feature,
        idf,
    ) in enumerate(
        zip(
            feature_names,
            vectorizer.idf_,
        )
    ):

        tokens = feature.split()

        contains_annotation = any(
            token.startswith(
                ANNOTATION_PREFIX
            )
            for token in tokens
        )

        records.append(
            {
                "feature_index":
                    feature_index,

                "feature":
                    feature,

                "ngram_size":
                    len(tokens),

                "feature_type":
                    classify_feature_type(
                        feature
                    ),

                "contains_annotation":
                    contains_annotation,

                "idf":
                    float(idf),
            }
        )

    return pd.DataFrame(
        records
    )

In [84]:
# Row mapping and saving helpers
def build_row_mapping(
    df: pd.DataFrame,
) -> pd.DataFrame:

    return pd.DataFrame(
        {
            "row_index":
                np.arange(
                    len(df)
                ),

            PARTICIPANT_ID_COLUMN:
                df[
                    PARTICIPANT_ID_COLUMN
                ].values,

            DATASET_COLUMN:
                df[
                    DATASET_COLUMN
                ].values,

            BINARY_TARGET_COLUMN:
                df[
                    BINARY_TARGET_COLUMN
                ].values,

            REGRESSION_TARGET_COLUMN:
                df[
                    REGRESSION_TARGET_COLUMN
                ].values,
        }
    )


def save_tfidf_base_files(
    output_dir: Path,
    vectorizer,
    vocabulary_df,
    X_train,
    X_dev,
    X_test,
    train_df,
    dev_df,
    test_df,
) -> None:

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    joblib.dump(
        vectorizer,
        output_dir
        / "vectorizer.joblib",
    )

    vocabulary_df.to_csv(
        output_dir
        / "vocabulary.csv",
        index=False,
    )

    save_npz(
        output_dir
        / "train.npz",
        X_train,
    )

    save_npz(
        output_dir
        / "dev.npz",
        X_dev,
    )

    save_npz(
        output_dir
        / "test.npz",
        X_test,
    )

    build_row_mapping(
        train_df
    ).to_csv(
        output_dir
        / "train_rows.csv",
        index=False,
    )

    build_row_mapping(
        dev_df
    ).to_csv(
        output_dir
        / "dev_rows.csv",
        index=False,
    )

    build_row_mapping(
        test_df
    ).to_csv(
        output_dir
        / "test_rows.csv",
        index=False,
    )

In [85]:
# Metrics
def calculate_regression_metrics(
    y_true,
    y_pred,
) -> dict:

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        spearman_result = spearmanr(
            y_true,
            y_pred,
        )

    return {
        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred,
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred,
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred,
                )
            ),

        "spearman_coefficient":
            float(
                spearman_result.statistic
            ),

        "spearman_p_value":
            float(
                spearman_result.pvalue
            ),
    }


def calculate_specificity(
    y_true,
    y_pred,
) -> float:

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    if (tn + fp) == 0:
        return np.nan

    return float(
        tn / (tn + fp)
    )


def calculate_classification_metrics(
    y_true,
    y_pred,
    y_probability,
) -> dict:

    try:

        roc_auc = roc_auc_score(
            y_true,
            y_probability,
        )

    except ValueError:

        roc_auc = np.nan

    return {
        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

        "f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0,
                )
            ),

        "precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0,
                )
            ),

        "recall_sensitivity":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    zero_division=0,
                )
            ),

        "specificity":
            calculate_specificity(
                y_true,
                y_pred,
            ),

        "roc_auc":
            float(roc_auc),
    }

In [86]:
# Feature-selection helpers
def feature_count_allowed(
    selected_count: int,
) -> bool:

    if (
        MIN_SELECTED_FEATURES
        is not None
        and selected_count
        < MIN_SELECTED_FEATURES
    ):
        return False

    if (
        MAX_SELECTED_FEATURES
        is not None
        and selected_count
        > MAX_SELECTED_FEATURES
    ):
        return False

    return True


def build_selected_features_dataframe(
    vocabulary_df: pd.DataFrame,
    coefficients,
    parameter_name: str,
    parameter_value: float,
) -> pd.DataFrame:

    coefficients = np.asarray(
        coefficients
    ).reshape(-1)

    selected_mask = (
        coefficients != 0
    )

    selected_df = (
        vocabulary_df.loc[
            selected_mask
        ]
        .copy()
    )

    selected_df[
        "coefficient"
    ] = coefficients[
        selected_mask
    ]

    selected_df[
        "absolute_coefficient"
    ] = np.abs(
        coefficients[
            selected_mask
        ]
    )

    selected_df[
        "coefficient_direction"
    ] = np.where(
        coefficients[
            selected_mask
        ] > 0,
        "positive",
        "negative",
    )

    selected_df[
        parameter_name
    ] = parameter_value

    selected_df = (
        selected_df
        .sort_values(
            by="absolute_coefficient",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    selected_df[
        "importance_rank"
    ] = (
        np.arange(
            len(selected_df)
        )
        + 1
    )

    return selected_df


def get_selected_matrices(
    X_train,
    X_dev,
    X_test,
    coefficients,
):

    coefficients = np.asarray(
        coefficients
    ).reshape(-1)

    selected_indices = np.flatnonzero(
        coefficients != 0
    )

    return (
        X_train[:, selected_indices],
        X_dev[:, selected_indices],
        X_test[:, selected_indices],
        selected_indices,
    )

In [87]:
# Data preparation
df = load_initial_dataset(
    INPUT_CSV_PATH
)

validate_initial_dataset(
    df
)

print(
    f"Total participants: {len(df)}"
)

print()

print(
    df[
        DATASET_COLUMN
    ]
    .value_counts()
    .sort_index()
)

df.head()

Total participants: 181

Dataset
dev       32
test      46
train    103
Name: count, dtype: int64


,Participant_ID,Folder,Audio_Filename,Output_Audio_Path,Transcript_Path,Dataset,Gender,PHQ8_Binary,PHQ8_Score
0,303,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P,303_AUDIO_removed_interviewer_manually_cleaned...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_...,train,0,0,0
1,304,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_P,304_AUDIO_removed_interviewer_manually_cleaned...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_...,train,0,0,6
2,305,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_P,305_AUDIO_removed_interviewer_manually_cleaned...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_...,train,1,0,7
3,310,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/310_P,310_AUDIO_removed_interviewer_manually_cleaned...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/310_...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/310_...,train,1,0,4
4,312,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/312_P,312_AUDIO_removed_interviewer_manually_cleaned...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/312_...,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/312_...,train,1,0,2


In [90]:
# Build participant documents
participant_documents_df = (
    build_participant_documents(
        df
    )
)

participant_documents_path = (
    TFIDF_OUTPUT_DIR
    / "participant_documents.csv"
)

participant_documents_df.to_csv(
    participant_documents_path,
    index=False,
)

print(
    f"Saved:\n"
    f"{participant_documents_path}"
)

participant_documents_df.head()

[1/181] Participant 303
[2/181] Participant 304
[3/181] Participant 305
[4/181] Participant 310
[5/181] Participant 312
[6/181] Participant 313
[7/181] Participant 315
[8/181] Participant 316
[9/181] Participant 317
[10/181] Participant 318
[11/181] Participant 319
[12/181] Participant 320
[13/181] Participant 322
[14/181] Participant 324
[15/181] Participant 325
[16/181] Participant 326
[17/181] Participant 327
[18/181] Participant 328
[19/181] Participant 330
[20/181] Participant 333
[21/181] Participant 336
[22/181] Participant 338
[23/181] Participant 339
[24/181] Participant 340
[25/181] Participant 341
[26/181] Participant 343
[27/181] Participant 344
[28/181] Participant 345
[29/181] Participant 347
[30/181] Participant 348
[31/181] Participant 350
[32/181] Participant 351
[33/181] Participant 352
[34/181] Participant 353
[35/181] Participant 355
[36/181] Participant 356
[37/181] Participant 357
[38/181] Participant 358
[39/181] Participant 360
[40/181] Participant 362
[41/181] 

,Participant_ID,Dataset,Transcript_Path,PHQ8_Binary,PHQ8_Score,participant_utterance_count,annotation_count,annotation_tokens,raw_participant_text,cleaned_participant_text
0,303,train,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_...,0,0,103,4,annotation_laughter; annotation_clears_throat;...,okay how 'bout yourself here in california yea...,okay how 'bout yourself here in california yea...
1,304,train,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_...,0,6,104,11,annotation_laughter; annotation_laughter; anno...,i'm doing good um from los angeles california ...,i'm doing good um from los angeles california ...
2,305,train,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_...,0,7,288,5,annotation_laughter; annotation_con; annotatio...,i'm doing alright uh originally i'm from calif...,i'm doing alright uh originally i'm from calif...
3,310,train,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/310_...,0,4,148,55,annotation_laughter; annotation_laughter; anno...,yes it's okay <laughter> fine <laughter> i liv...,yes it's okay annotation_laughter fine annotat...
4,312,train,/Volumes/LaCie/DAIC-WOZ/Processed Dataset/312_...,0,2,126,11,annotation_ma; annotation_sigh; annotation_amo...,yes fine how about you here yes the weather we...,yes fine how about you here yes the weather we...


In [91]:
participant_documents_df[
    [
        PARTICIPANT_ID_COLUMN,
        "participant_utterance_count",
        "annotation_count",
        "annotation_tokens",
        "raw_participant_text",
        "cleaned_participant_text",
    ]
].head(10)

,Participant_ID,participant_utterance_count,annotation_count,annotation_tokens,raw_participant_text,cleaned_participant_text
0,303,103,4,annotation_laughter; annotation_clears_throat;...,okay how 'bout yourself here in california yea...,okay how 'bout yourself here in california yea...
1,304,104,11,annotation_laughter; annotation_laughter; anno...,i'm doing good um from los angeles california ...,i'm doing good um from los angeles california ...
2,305,288,5,annotation_laughter; annotation_con; annotatio...,i'm doing alright uh originally i'm from calif...,i'm doing alright uh originally i'm from calif...
3,310,148,55,annotation_laughter; annotation_laughter; anno...,yes it's okay <laughter> fine <laughter> i liv...,yes it's okay annotation_laughter fine annotat...
4,312,126,11,annotation_ma; annotation_sigh; annotation_amo...,yes fine how about you here yes the weather we...,yes fine how about you here yes the weather we...
5,313,114,6,annotation_cough; annotation_sniffle; annotati...,sure uh okay i guess not bad not good so so de...,sure uh okay i guess not bad not good so so de...
6,315,135,18,annotation_laughter; annotation_sigh; annotati...,alright yes okay and you inglewood california ...,alright yes okay and you inglewood california ...
7,316,136,16,annotation_sneeze; annotation_laughter; annota...,yes i'm fine new york uh for business now um i...,yes i'm fine new york uh for business now um i...
8,317,112,16,annotation_clears_throat; annotation_laughter;...,yes uh i'm okay a little sleepy right now i'...,yes uh i'm okay a little sleepy right now i'm ...
9,318,68,16,annotation_sigh; annotation_laughter; annotati...,yes i'm alright los angeles california yep ...,yes i'm alright los angeles california yep um ...


In [92]:
participant_documents_df.loc[
    participant_documents_df[
        "annotation_count"
    ] > 0,
    [
        PARTICIPANT_ID_COLUMN,
        "annotation_count",
        "annotation_tokens",
        "raw_participant_text",
        "cleaned_participant_text",
    ],
].head(20)

,Participant_ID,annotation_count,annotation_tokens,raw_participant_text,cleaned_participant_text
0,303,4,annotation_laughter; annotation_clears_throat;...,okay how 'bout yourself here in california yea...,okay how 'bout yourself here in california yea...
1,304,11,annotation_laughter; annotation_laughter; anno...,i'm doing good um from los angeles california ...,i'm doing good um from los angeles california ...
2,305,5,annotation_laughter; annotation_con; annotatio...,i'm doing alright uh originally i'm from calif...,i'm doing alright uh originally i'm from calif...
3,310,55,annotation_laughter; annotation_laughter; anno...,yes it's okay <laughter> fine <laughter> i liv...,yes it's okay annotation_laughter fine annotat...
4,312,11,annotation_ma; annotation_sigh; annotation_amo...,yes fine how about you here yes the weather we...,yes fine how about you here yes the weather we...
5,313,6,annotation_cough; annotation_sniffle; annotati...,sure uh okay i guess not bad not good so so de...,sure uh okay i guess not bad not good so so de...
6,315,18,annotation_laughter; annotation_sigh; annotati...,alright yes okay and you inglewood california ...,alright yes okay and you inglewood california ...
7,316,16,annotation_sneeze; annotation_laughter; annota...,yes i'm fine new york uh for business now um i...,yes i'm fine new york uh for business now um i...
8,317,16,annotation_clears_throat; annotation_laughter;...,yes uh i'm okay a little sleepy right now i'...,yes uh i'm okay a little sleepy right now i'm ...
9,318,16,annotation_sigh; annotation_laughter; annotati...,yes i'm alright los angeles california yep ...,yes i'm alright los angeles california yep um ...


In [5]:
participant_documents_df = pd.read_csv("/Users/sonalprasad/Documents/Thesis/TFIDF/participant_documents.csv")

In [8]:
train_df = (
    participant_documents_df.loc[
        participant_documents_df[
            DATASET_COLUMN
        ] == "train"
    ]
    .reset_index(drop=True)
)

dev_df = (
    participant_documents_df.loc[
        participant_documents_df[
            DATASET_COLUMN
        ] == "dev"
    ]
    .reset_index(drop=True)
)

test_df = (
    participant_documents_df.loc[
        participant_documents_df[
            DATASET_COLUMN
        ] == "test"
    ]
    .reset_index(drop=True)
)


print(
    f"Train: {len(train_df)}"
)

print(
    f"Dev:   {len(dev_df)}"
)

print(
    f"Test:  {len(test_df)}"
)

Train: 103
Dev:   32
Test:  46


In [9]:
# How many unigram features are there with the same settings as Experiment 1?

from sklearn.feature_extraction.text import TfidfVectorizer

temp_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.9,
    max_features=None,
)

temp_vectorizer.fit(
    train_df["cleaned_participant_text"]
)

feature_names = temp_vectorizer.get_feature_names_out()

print(
    "Number of eligible unique unigrams in Train:",
    len(feature_names)
)

Number of eligible unique unigrams in Train: 2989


In [13]:
# How many unique unigram features exist across the train DAIC-WOZ corpus?

# Train transcripts only
train_text = (
    train_df["cleaned_participant_text"]
    .fillna("")
    .astype(str)
)

# Extract all unique unigrams
vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0,
    max_features=None,
)

vectorizer.fit(train_text)

unique_unigrams = vectorizer.get_feature_names_out()

print(f"Number of unique unigram features: {len(unique_unigrams):,}")

Number of unique unigram features: 6,037


In [10]:
# How many unique unigram features exist across the entire DAIC-WOZ corpus?

all_text = pd.concat([
    train_df["cleaned_participant_text"],
    dev_df["cleaned_participant_text"],
    test_df["cleaned_participant_text"],
]).dropna()

temp_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0,
    max_features=None,
)

temp_vectorizer.fit(all_text)

feature_names = temp_vectorizer.get_feature_names_out()

print(
    "Total unique unigrams across Train + Dev + Test:",
    len(feature_names)
)

Total unique unigrams across Train + Dev + Test: 8307


In [12]:
# How many unigram and bigram features are there with the same settings as Experiment 1?

# Use Train only
train_text = (
    train_df["cleaned_participant_text"]
    .fillna("")
    .astype(str)
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    max_features=None,
)

vectorizer.fit(train_text)

features = vectorizer.get_feature_names_out()

# Separate unigrams and bigrams
unigrams = [
    feature
    for feature in features
    if len(feature.split()) == 1
]

bigrams = [
    feature
    for feature in features
    if len(feature.split()) == 2
]

print(f"Eligible unigrams: {len(unigrams):,}")
print(f"Eligible bigrams:  {len(bigrams):,}")
print(f"Total features:    {len(features):,}")

Eligible unigrams: 2,989
Eligible bigrams:  13,380
Total features:    16,369


In [14]:
# How many unique unigram and unique bigram features exist across the train DAIC-WOZ corpus?

# Train transcripts only
train_text = (
    train_df["cleaned_participant_text"]
    .fillna("")
    .astype(str)
)

# -------------------------
# Unique unigrams
# -------------------------

unigram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0,
    max_features=None,
)

unigram_vectorizer.fit(train_text)

unique_unigrams = (
    unigram_vectorizer
    .get_feature_names_out()
)


# -------------------------
# Unique bigrams
# -------------------------

bigram_vectorizer = TfidfVectorizer(
    ngram_range=(2, 2),
    min_df=1,
    max_df=1.0,
    max_features=None,
)

bigram_vectorizer.fit(train_text)

unique_bigrams = (
    bigram_vectorizer
    .get_feature_names_out()
)


# -------------------------
# Results
# -------------------------

print(
    f"Unique unigram features: "
    f"{len(unique_unigrams):,}"
)

print(
    f"Unique bigram features: "
    f"{len(unique_bigrams):,}"
)

print(
    f"Total unigram + bigram features: "
    f"{len(unique_unigrams) + len(unique_bigrams):,}"
)

Unique unigram features: 6,037
Unique bigram features: 51,015
Total unigram + bigram features: 57,052


In [11]:
# How many unique unigram and unique bigram features exist across the entire DAIC-WOZ corpus?

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0,
    max_features=None,
)

vectorizer.fit(all_text)

features = vectorizer.get_feature_names_out()

unigrams = [
    feature for feature in features
    if len(feature.split()) == 1
]

bigrams = [
    feature for feature in features
    if len(feature.split()) == 2
]

print(f"Unique unigrams: {len(unigrams):,}")
print(f"Unique bigrams:  {len(bigrams):,}")
print(f"Total:           {len(features):,}")

Unique unigrams: 8,307
Unique bigrams:  80,698
Total:           89,005


### Experiment 1 — 3,000 unigrams

In [94]:
EXPERIMENT_1_DIR = (
    TFIDF_OUTPUT_DIR
    / "unigram_3000"
)

(
    vectorizer_unigram_3000,
    X_train_unigram_3000,
    X_dev_unigram_3000,
    X_test_unigram_3000,
) = fit_tfidf(
    train_df=train_df,
    dev_df=dev_df,
    test_df=test_df,
    ngram_range=(1, 1),
    max_features=3000,
)

vocabulary_unigram_3000 = (
    build_vocabulary_dataframe(
        vectorizer_unigram_3000
    )
)

save_tfidf_base_files(
    output_dir=
        EXPERIMENT_1_DIR,

    vectorizer=
        vectorizer_unigram_3000,

    vocabulary_df=
        vocabulary_unigram_3000,

    X_train=
        X_train_unigram_3000,

    X_dev=
        X_dev_unigram_3000,

    X_test=
        X_test_unigram_3000,

    train_df=
        train_df,

    dev_df=
        dev_df,

    test_df=
        test_df,
)

print(
    "Vocabulary size:",
    len(
        vocabulary_unigram_3000
    )
)

print(
    "Train shape:",
    X_train_unigram_3000.shape
)

print(
    "Dev shape:",
    X_dev_unigram_3000.shape
)

print(
    "Test shape:",
    X_test_unigram_3000.shape
)

vocabulary_unigram_3000.head(25)

Vocabulary size: 3000
Train shape: (103, 3000)
Dev shape: (32, 3000)
Test shape: (46, 3000)


,feature_index,feature,ngram_size,feature_type,contains_annotation,idf
0,0,a_d_h_d,1,unigram_word,False,4.258097
1,1,a_m,1,unigram_word,False,4.545779
2,2,a_p,1,unigram_word,False,4.545779
3,3,abilities,1,unigram_word,False,4.545779
4,4,ability,1,unigram_word,False,4.034953
5,5,able,1,unigram_word,False,1.815750
6,6,above,1,unigram_word,False,3.564949
7,7,absolutely,1,unigram_word,False,4.258097
8,8,abuse,1,unigram_word,False,4.034953
9,9,abused,1,unigram_word,False,4.258097


### Experiment 2 — 3,000 unigrams + L1

In [95]:
# Prepare Experiment 2

EXPERIMENT_2_DIR = (
    TFIDF_OUTPUT_DIR
    / "unigram_3000_l1"
)

EXPERIMENT_2_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

vectorizer_unigram_3000_l1 = (
    vectorizer_unigram_3000
)

X_train_unigram_3000_l1 = (
    X_train_unigram_3000
)

X_dev_unigram_3000_l1 = (
    X_dev_unigram_3000
)

X_test_unigram_3000_l1 = (
    X_test_unigram_3000
)

vocabulary_unigram_3000_l1 = (
    vocabulary_unigram_3000.copy()
)

save_tfidf_base_files(
    output_dir=
        EXPERIMENT_2_DIR,

    vectorizer=
        vectorizer_unigram_3000_l1,

    vocabulary_df=
        vocabulary_unigram_3000_l1,

    X_train=
        X_train_unigram_3000_l1,

    X_dev=
        X_dev_unigram_3000_l1,

    X_test=
        X_test_unigram_3000_l1,

    train_df=
        train_df,

    dev_df=
        dev_df,

    test_df=
        test_df,
)

In [96]:
# Prepare targets

y_train_phq8 = (
    train_df[
        REGRESSION_TARGET_COLUMN
    ]
    .astype(float)
    .to_numpy()
)

y_dev_phq8 = (
    dev_df[
        REGRESSION_TARGET_COLUMN
    ]
    .astype(float)
    .to_numpy()
)

y_test_phq8 = (
    test_df[
        REGRESSION_TARGET_COLUMN
    ]
    .astype(float)
    .to_numpy()
)


y_train_binary = (
    train_df[
        BINARY_TARGET_COLUMN
    ]
    .astype(int)
    .to_numpy()
)

y_dev_binary = (
    dev_df[
        BINARY_TARGET_COLUMN
    ]
    .astype(int)
    .to_numpy()
)

y_test_binary = (
    test_df[
        BINARY_TARGET_COLUMN
    ]
    .astype(int)
    .to_numpy()
)

### PHQ-8 LASSO Search for Experiment 2A (Regression)

In [97]:
regression_results = []

best_lasso_model = None

best_lasso_alpha = None

best_dev_mae = np.inf


for alpha in REGRESSION_ALPHAS:

    model = Lasso(
        alpha=alpha,
        max_iter=LASSO_MAX_ITER,
        tol=LASSO_TOLERANCE,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_train_unigram_3000_l1,
        y_train_phq8,
    )

    selected_count = int(
        np.sum(
            model.coef_ != 0
        )
    )

    train_predictions = (
        model.predict(
            X_train_unigram_3000_l1
        )
    )

    dev_predictions = (
        model.predict(
            X_dev_unigram_3000_l1
        )
    )

    train_metrics = (
        calculate_regression_metrics(
            y_train_phq8,
            train_predictions,
        )
    )

    dev_metrics = (
        calculate_regression_metrics(
            y_dev_phq8,
            dev_predictions,
        )
    )

    allowed = feature_count_allowed(
        selected_count
    )

    regression_results.append(
        {
            "alpha":
                alpha,

            "selected_features":
                selected_count,

            "feature_count_allowed":
                allowed,

            "train_mae":
                train_metrics["mae"],

            "train_rmse":
                train_metrics["rmse"],

            "train_r2":
                train_metrics["r2"],

            "train_spearman_coefficient":
                train_metrics[
                    "spearman_coefficient"
                ],

            "train_spearman_p_value":
                train_metrics[
                    "spearman_p_value"
                ],

            "dev_mae":
                dev_metrics["mae"],

            "dev_rmse":
                dev_metrics["rmse"],

            "dev_r2":
                dev_metrics["r2"],

            "dev_spearman_coefficient":
                dev_metrics[
                    "spearman_coefficient"
                ],

            "dev_spearman_p_value":
                dev_metrics[
                    "spearman_p_value"
                ],
        }
    )

    if (
        allowed
        and selected_count > 0
        and dev_metrics["mae"]
        < best_dev_mae
    ):

        best_dev_mae = (
            dev_metrics["mae"]
        )

        best_lasso_alpha = alpha

        best_lasso_model = model


regression_search_df = (
    pd.DataFrame(
        regression_results
    )
)

regression_search_df

,alpha,selected_features,feature_count_allowed,train_mae,train_rmse,train_r2,train_spearman_coefficient,train_spearman_p_value,dev_mae,dev_rmse,dev_r2,dev_spearman_coefficient,dev_spearman_p_value
0,0.0001,109,True,0.023274,0.027127,0.999974,0.996545,5.662664e-111,5.177780,6.621832,-0.094933,0.241957,0.182143
1,0.0005,101,True,0.113171,0.132975,0.999374,0.996545,5.662664e-111,5.112654,6.644362,-0.102397,0.277994,0.123426
2,0.0010,100,True,0.225326,0.265170,0.997509,0.996545,5.662664e-111,5.082318,6.619789,-0.094258,0.273581,0.129745
3,0.0025,89,True,0.555446,0.656307,0.984739,0.994666,1.815246e-101,4.948610,6.512713,-0.059145,0.306676,0.087783
4,0.0050,79,True,1.068230,1.268411,0.942999,0.985067,5.454678e-79,4.990342,6.503865,-0.056269,0.293438,0.103101
5,0.0100,65,True,1.964587,2.368065,0.801322,0.939640,7.640276e-49,5.301999,6.506133,-0.057006,0.276707,0.125245
6,0.0250,14,True,3.660150,4.441777,0.301003,0.668270,1.238016e-14,5.155055,6.301787,0.008349,0.318259,0.075865
7,0.0500,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN
8,0.1000,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN
9,0.2500,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN


In [98]:
# Save experiment 2A

if best_lasso_model is None:
    raise RuntimeError(
        "No valid LASSO model found."
    )


EXPERIMENT_2_PHQ8_DIR = (
    EXPERIMENT_2_DIR
    / "phq8_regression"
)

EXPERIMENT_2_PHQ8_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


test_predictions = (
    best_lasso_model.predict(
        X_test_unigram_3000_l1
    )
)

test_metrics = (
    calculate_regression_metrics(
        y_test_phq8,
        test_predictions,
    )
)


print(
    "Best alpha:",
    best_lasso_alpha
)

print(
    "Selected features:",
    np.sum(
        best_lasso_model.coef_
        != 0
    )
)

print(
    "Best dev MAE:",
    best_dev_mae
)

print(
    "Test metrics:"
)

print(
    test_metrics
)


regression_search_df.to_csv(
    EXPERIMENT_2_PHQ8_DIR
    / "regularization_search.csv",
    index=False,
)


joblib.dump(
    best_lasso_model,
    EXPERIMENT_2_PHQ8_DIR
    / "lasso_model.joblib",
)


selected_phq8_features = (
    build_selected_features_dataframe(
        vocabulary_df=
            vocabulary_unigram_3000_l1,

        coefficients=
            best_lasso_model.coef_,

        parameter_name=
            "alpha",

        parameter_value=
            best_lasso_alpha,
    )
)


selected_phq8_features.to_csv(
    EXPERIMENT_2_PHQ8_DIR
    / "selected_features.csv",
    index=False,
)


(
    X_train_phq8_selected,
    X_dev_phq8_selected,
    X_test_phq8_selected,
    phq8_selected_indices,
) = get_selected_matrices(
    X_train_unigram_3000_l1,
    X_dev_unigram_3000_l1,
    X_test_unigram_3000_l1,
    best_lasso_model.coef_,
)


save_npz(
    EXPERIMENT_2_PHQ8_DIR
    / "train_selected.npz",
    X_train_phq8_selected,
)

save_npz(
    EXPERIMENT_2_PHQ8_DIR
    / "dev_selected.npz",
    X_dev_phq8_selected,
)

save_npz(
    EXPERIMENT_2_PHQ8_DIR
    / "test_selected.npz",
    X_test_phq8_selected,
)


np.save(
    EXPERIMENT_2_PHQ8_DIR
    / "selected_feature_indices.npy",
    phq8_selected_indices,
)


selected_phq8_features.head(30)

Best alpha: 0.0025
Selected features: 89
Best dev MAE: 4.948610444170197
Test metrics:
{'mae': 5.329482306506329, 'rmse': 6.644282312826478, 'r2': -0.07420530409691772, 'spearman_coefficient': 0.2457307794553044, 'spearman_p_value': 0.09974120800980846}


,feature_index,feature,ngram_size,feature_type,contains_annotation,idf,coefficient,absolute_coefficient,coefficient_direction,alpha,importance_rank
0,2573,sync,1,unigram_word,False,4.258097,56.536021,56.536021,positive,0.0025,1
1,1560,learn,1,unigram_word,False,2.277095,-45.995013,45.995013,negative,0.0025,2
2,1042,february,1,unigram_word,False,4.258097,40.836019,40.836019,positive,0.0025,3
3,1245,guilty,1,unigram_word,False,2.508897,-40.490783,40.490783,negative,0.0025,4
4,1658,management,1,unigram_word,False,3.447166,-26.015617,26.015617,negative,0.0025,5
5,2189,relationship,1,unigram_word,False,2.089043,-24.697586,24.697586,negative,0.0025,6
6,761,decided,1,unigram_word,False,2.386294,24.505352,24.505352,positive,0.0025,7
7,358,behavior,1,unigram_word,False,3.246496,-23.969572,23.969572,negative,0.0025,8
8,1216,graduated,1,unigram_word,False,3.079442,23.362865,23.362865,positive,0.0025,9
9,2653,thinking,1,unigram_word,False,2.089043,23.214517,23.214517,positive,0.0025,10


### Experiment 2B — Binary L1 Logistic Search

In [99]:
classification_results = []

best_logistic_model = None

best_logistic_C = None

best_dev_balanced_accuracy = (
    -np.inf
)


for C in CLASSIFICATION_C_VALUES:

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=C,
        class_weight=
            LOGISTIC_CLASS_WEIGHT,
        max_iter=
            LOGISTIC_MAX_ITER,
        random_state=
            RANDOM_STATE,
    )

    model.fit(
        X_train_unigram_3000_l1,
        y_train_binary,
    )

    coefficients = (
        model.coef_[0]
    )

    selected_count = int(
        np.sum(
            coefficients != 0
        )
    )

    train_predictions = (
        model.predict(
            X_train_unigram_3000_l1
        )
    )

    train_probabilities = (
        model.predict_proba(
            X_train_unigram_3000_l1
        )[:, 1]
    )

    dev_predictions = (
        model.predict(
            X_dev_unigram_3000_l1
        )
    )

    dev_probabilities = (
        model.predict_proba(
            X_dev_unigram_3000_l1
        )[:, 1]
    )

    train_metrics = (
        calculate_classification_metrics(
            y_train_binary,
            train_predictions,
            train_probabilities,
        )
    )

    dev_metrics = (
        calculate_classification_metrics(
            y_dev_binary,
            dev_predictions,
            dev_probabilities,
        )
    )

    allowed = feature_count_allowed(
        selected_count
    )

    classification_results.append(
        {
            "C":
                C,

            "selected_features":
                selected_count,

            "feature_count_allowed":
                allowed,

            "train_balanced_accuracy":
                train_metrics[
                    "balanced_accuracy"
                ],

            "train_accuracy":
                train_metrics[
                    "accuracy"
                ],

            "train_f1":
                train_metrics[
                    "f1"
                ],

            "train_precision":
                train_metrics[
                    "precision"
                ],

            "train_recall_sensitivity":
                train_metrics[
                    "recall_sensitivity"
                ],

            "train_specificity":
                train_metrics[
                    "specificity"
                ],

            "train_roc_auc":
                train_metrics[
                    "roc_auc"
                ],

            "dev_balanced_accuracy":
                dev_metrics[
                    "balanced_accuracy"
                ],

            "dev_accuracy":
                dev_metrics[
                    "accuracy"
                ],

            "dev_f1":
                dev_metrics[
                    "f1"
                ],

            "dev_precision":
                dev_metrics[
                    "precision"
                ],

            "dev_recall_sensitivity":
                dev_metrics[
                    "recall_sensitivity"
                ],

            "dev_specificity":
                dev_metrics[
                    "specificity"
                ],

            "dev_roc_auc":
                dev_metrics[
                    "roc_auc"
                ],
        }
    )

    if (
        allowed
        and selected_count > 0
        and dev_metrics[
            "balanced_accuracy"
        ] > best_dev_balanced_accuracy
    ):

        best_dev_balanced_accuracy = (
            dev_metrics[
                "balanced_accuracy"
            ]
        )

        best_logistic_C = C

        best_logistic_model = model


classification_search_df = (
    pd.DataFrame(
        classification_results
    )
)

classification_search_df

/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

,C,selected_features,feature_count_allowed,train_balanced_accuracy,train_accuracy,train_f1,train_precision,train_recall_sensitivity,train_specificity,train_roc_auc,dev_balanced_accuracy,dev_accuracy,dev_f1,dev_precision,dev_recall_sensitivity,dev_specificity,dev_roc_auc
0,0.0010,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
1,0.0025,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
2,0.0050,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
3,0.0100,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
4,0.0250,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
5,0.0500,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
6,0.1000,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
7,0.2500,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
8,0.5000,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
9,1.0000,0,True,0.500000,0.718447,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000


In [100]:
# Save experiment 2B

if best_logistic_model is None:
    raise RuntimeError(
        "No valid logistic regression model found."
    )


EXPERIMENT_2_BINARY_DIR = (
    EXPERIMENT_2_DIR
    / "depression_binary"
)

EXPERIMENT_2_BINARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


test_predictions = (
    best_logistic_model.predict(
        X_test_unigram_3000_l1
    )
)

test_probabilities = (
    best_logistic_model.predict_proba(
        X_test_unigram_3000_l1
    )[:, 1]
)

test_metrics = (
    calculate_classification_metrics(
        y_test_binary,
        test_predictions,
        test_probabilities,
    )
)


print(
    "Best C:",
    best_logistic_C
)

print(
    "Selected features:",
    np.sum(
        best_logistic_model.coef_[0]
        != 0
    )
)

print(
    "Best dev balanced accuracy:",
    best_dev_balanced_accuracy
)

print(
    "Test metrics:"
)

print(
    test_metrics
)


classification_search_df.to_csv(
    EXPERIMENT_2_BINARY_DIR
    / "regularization_search.csv",
    index=False,
)


joblib.dump(
    best_logistic_model,
    EXPERIMENT_2_BINARY_DIR
    / "l1_logistic_model.joblib",
)


selected_binary_features = (
    build_selected_features_dataframe(
        vocabulary_df=
            vocabulary_unigram_3000_l1,

        coefficients=
            best_logistic_model.coef_[0],

        parameter_name=
            "C",

        parameter_value=
            best_logistic_C,
    )
)


selected_binary_features.to_csv(
    EXPERIMENT_2_BINARY_DIR
    / "selected_features.csv",
    index=False,
)


(
    X_train_binary_selected,
    X_dev_binary_selected,
    X_test_binary_selected,
    binary_selected_indices,
) = get_selected_matrices(
    X_train_unigram_3000_l1,
    X_dev_unigram_3000_l1,
    X_test_unigram_3000_l1,
    best_logistic_model.coef_[0],
)


save_npz(
    EXPERIMENT_2_BINARY_DIR
    / "train_selected.npz",
    X_train_binary_selected,
)

save_npz(
    EXPERIMENT_2_BINARY_DIR
    / "dev_selected.npz",
    X_dev_binary_selected,
)

save_npz(
    EXPERIMENT_2_BINARY_DIR
    / "test_selected.npz",
    X_test_binary_selected,
)


np.save(
    EXPERIMENT_2_BINARY_DIR
    / "selected_feature_indices.npy",
    binary_selected_indices,
)


selected_binary_features.head(30)

Best C: 2.5
Selected features: 1
Best dev balanced accuracy: 0.7034632034632035
Test metrics:
{'balanced_accuracy': 0.5089285714285714, 'accuracy': 0.6521739130434783, 'f1': 0.2, 'precision': 0.3333333333333333, 'recall_sensitivity': 0.14285714285714285, 'specificity': 0.875, 'roc_auc': 0.5111607142857143}


,feature_index,feature,ngram_size,feature_type,contains_annotation,idf,coefficient,absolute_coefficient,coefficient_direction,C,importance_rank
0,2639,therapist,1,unigram_word,False,2.936341,4.239029,4.239029,positive,2.5,1


### Experiment 3 — 3,000 unigrams + bigrams

In [101]:
EXPERIMENT_3_DIR = (
    TFIDF_OUTPUT_DIR
    / "unigram_bigram_3000"
)


(
    vectorizer_unigram_bigram_3000,
    X_train_unigram_bigram_3000,
    X_dev_unigram_bigram_3000,
    X_test_unigram_bigram_3000,
) = fit_tfidf(
    train_df=train_df,
    dev_df=dev_df,
    test_df=test_df,
    ngram_range=(1, 2),
    max_features=3000,
)


vocabulary_unigram_bigram_3000 = (
    build_vocabulary_dataframe(
        vectorizer_unigram_bigram_3000
    )
)


save_tfidf_base_files(
    output_dir=
        EXPERIMENT_3_DIR,

    vectorizer=
        vectorizer_unigram_bigram_3000,

    vocabulary_df=
        vocabulary_unigram_bigram_3000,

    X_train=
        X_train_unigram_bigram_3000,

    X_dev=
        X_dev_unigram_bigram_3000,

    X_test=
        X_test_unigram_bigram_3000,

    train_df=
        train_df,

    dev_df=
        dev_df,

    test_df=
        test_df,
)


print(
    "Vocabulary size:",
    len(
        vocabulary_unigram_bigram_3000
    )
)

print(
    "Train:",
    X_train_unigram_bigram_3000.shape
)

print(
    "Dev:",
    X_dev_unigram_bigram_3000.shape
)

print(
    "Test:",
    X_test_unigram_bigram_3000.shape
)

print()

print(
    vocabulary_unigram_bigram_3000[
        "feature_type"
    ].value_counts()
)

Vocabulary size: 3000
Train: (103, 3000)
Dev: (32, 3000)
Test: (46, 3000)

feature_type
bigram_word_word          1959
unigram_word               977
bigram_with_annotation      45
unigram_annotation          19
Name: count, dtype: int64


### Experiment 4 — Large unigram + bigram candidate pool

In [102]:
EXPERIMENT_4_MAX_FEATURES = 10_000

In [103]:
EXPERIMENT_4_DIR = (
    TFIDF_OUTPUT_DIR
    / "unigram_bigram_l1_candidates"
)


(
    vectorizer_unigram_bigram_l1,
    X_train_unigram_bigram_l1,
    X_dev_unigram_bigram_l1,
    X_test_unigram_bigram_l1,
) = fit_tfidf(
    train_df=train_df,
    dev_df=dev_df,
    test_df=test_df,
    ngram_range=(1, 2),
    max_features=
        EXPERIMENT_4_MAX_FEATURES,
)


vocabulary_unigram_bigram_l1 = (
    build_vocabulary_dataframe(
        vectorizer_unigram_bigram_l1
    )
)


save_tfidf_base_files(
    output_dir=
        EXPERIMENT_4_DIR,

    vectorizer=
        vectorizer_unigram_bigram_l1,

    vocabulary_df=
        vocabulary_unigram_bigram_l1,

    X_train=
        X_train_unigram_bigram_l1,

    X_dev=
        X_dev_unigram_bigram_l1,

    X_test=
        X_test_unigram_bigram_l1,

    train_df=
        train_df,

    dev_df=
        dev_df,

    test_df=
        test_df,
)


print(
    "Vocabulary size:",
    len(
        vocabulary_unigram_bigram_l1
    )
)

print()

print(
    vocabulary_unigram_bigram_l1[
        "feature_type"
    ].value_counts()
)

Vocabulary size: 10000

feature_type
bigram_word_word          7467
unigram_word              2232
bigram_with_annotation     239
unigram_annotation          62
Name: count, dtype: int64


### Experiment 4A: PHQ-8 LASSO search

In [104]:
experiment4_regression_results = []

experiment4_best_lasso_model = None

experiment4_best_alpha = None

experiment4_best_dev_mae = np.inf


for alpha in REGRESSION_ALPHAS:

    model = Lasso(
        alpha=alpha,
        max_iter=LASSO_MAX_ITER,
        tol=LASSO_TOLERANCE,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_train_unigram_bigram_l1,
        y_train_phq8,
    )

    selected_count = int(
        np.sum(
            model.coef_ != 0
        )
    )

    train_predictions = (
        model.predict(
            X_train_unigram_bigram_l1
        )
    )

    dev_predictions = (
        model.predict(
            X_dev_unigram_bigram_l1
        )
    )

    train_metrics = (
        calculate_regression_metrics(
            y_train_phq8,
            train_predictions,
        )
    )

    dev_metrics = (
        calculate_regression_metrics(
            y_dev_phq8,
            dev_predictions,
        )
    )

    allowed = feature_count_allowed(
        selected_count
    )

    experiment4_regression_results.append(
        {
            "alpha":
                alpha,

            "selected_features":
                selected_count,

            "feature_count_allowed":
                allowed,

            "train_mae":
                train_metrics["mae"],

            "train_rmse":
                train_metrics["rmse"],

            "train_r2":
                train_metrics["r2"],

            "train_spearman_coefficient":
                train_metrics[
                    "spearman_coefficient"
                ],

            "train_spearman_p_value":
                train_metrics[
                    "spearman_p_value"
                ],

            "dev_mae":
                dev_metrics["mae"],

            "dev_rmse":
                dev_metrics["rmse"],

            "dev_r2":
                dev_metrics["r2"],

            "dev_spearman_coefficient":
                dev_metrics[
                    "spearman_coefficient"
                ],

            "dev_spearman_p_value":
                dev_metrics[
                    "spearman_p_value"
                ],
        }
    )

    if (
        allowed
        and selected_count > 0
        and dev_metrics["mae"]
        < experiment4_best_dev_mae
    ):

        experiment4_best_dev_mae = (
            dev_metrics["mae"]
        )

        experiment4_best_alpha = alpha

        experiment4_best_lasso_model = (
            model
        )


experiment4_regression_search_df = (
    pd.DataFrame(
        experiment4_regression_results
    )
)

experiment4_regression_search_df

,alpha,selected_features,feature_count_allowed,train_mae,train_rmse,train_r2,train_spearman_coefficient,train_spearman_p_value,dev_mae,dev_rmse,dev_r2,dev_spearman_coefficient,dev_spearman_p_value
0,0.0001,106,True,0.033917,0.040093,0.999943,0.996545,5.662664e-111,5.196250,6.313400,0.004691,0.326164,0.068479
1,0.0005,98,True,0.166259,0.198117,0.998609,0.996545,5.662664e-111,5.173677,6.231724,0.030277,0.367349,0.038612
2,0.0010,94,True,0.329254,0.391424,0.994572,0.996413,3.763269e-110,5.210845,6.259153,0.021722,0.344550,0.053468
3,0.0025,80,True,0.803956,0.950734,0.967976,0.989393,1.910561e-86,5.176109,6.196683,0.041152,0.372497,0.035768
4,0.0050,59,True,1.535384,1.812709,0.883583,0.964894,1.859873e-60,5.034026,6.127829,0.062342,0.451372,0.009513
5,0.0100,33,True,2.733857,3.213850,0.634057,0.885786,1.893899e-35,4.955494,6.124282,0.063427,0.432618,0.013399
6,0.0250,2,True,4.292866,5.212452,0.037400,0.419683,1.020094e-05,5.403311,6.441363,-0.036065,-0.030208,0.869634
7,0.0500,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN
8,0.1000,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN
9,0.2500,0,True,4.378547,5.312747,0.000000,NaN,NaN,5.375000,6.398846,-0.022433,NaN,NaN


In [105]:
# Save Experiment 4A

if experiment4_best_lasso_model is None:
    raise RuntimeError(
        "No valid Experiment 4 LASSO model found."
    )


EXPERIMENT_4_PHQ8_DIR = (
    EXPERIMENT_4_DIR
    / "phq8_regression"
)

EXPERIMENT_4_PHQ8_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


test_predictions = (
    experiment4_best_lasso_model.predict(
        X_test_unigram_bigram_l1
    )
)

test_metrics = (
    calculate_regression_metrics(
        y_test_phq8,
        test_predictions,
    )
)


print(
    "Best alpha:",
    experiment4_best_alpha
)

print(
    "Selected features:",
    np.sum(
        experiment4_best_lasso_model.coef_
        != 0
    )
)

print(
    "Best dev MAE:",
    experiment4_best_dev_mae
)

print(
    "Test metrics:"
)

print(
    test_metrics
)


experiment4_regression_search_df.to_csv(
    EXPERIMENT_4_PHQ8_DIR
    / "regularization_search.csv",
    index=False,
)


joblib.dump(
    experiment4_best_lasso_model,
    EXPERIMENT_4_PHQ8_DIR
    / "lasso_model.joblib",
)


experiment4_selected_phq8_features = (
    build_selected_features_dataframe(
        vocabulary_df=
            vocabulary_unigram_bigram_l1,

        coefficients=
            experiment4_best_lasso_model.coef_,

        parameter_name=
            "alpha",

        parameter_value=
            experiment4_best_alpha,
    )
)


experiment4_selected_phq8_features.to_csv(
    EXPERIMENT_4_PHQ8_DIR
    / "selected_features.csv",
    index=False,
)


(
    X_train_exp4_phq8_selected,
    X_dev_exp4_phq8_selected,
    X_test_exp4_phq8_selected,
    exp4_phq8_selected_indices,
) = get_selected_matrices(
    X_train_unigram_bigram_l1,
    X_dev_unigram_bigram_l1,
    X_test_unigram_bigram_l1,
    experiment4_best_lasso_model.coef_,
)


save_npz(
    EXPERIMENT_4_PHQ8_DIR
    / "train_selected.npz",
    X_train_exp4_phq8_selected,
)

save_npz(
    EXPERIMENT_4_PHQ8_DIR
    / "dev_selected.npz",
    X_dev_exp4_phq8_selected,
)

save_npz(
    EXPERIMENT_4_PHQ8_DIR
    / "test_selected.npz",
    X_test_exp4_phq8_selected,
)


np.save(
    EXPERIMENT_4_PHQ8_DIR
    / "selected_feature_indices.npy",
    exp4_phq8_selected_indices,
)


experiment4_selected_phq8_features.head(30)

Best alpha: 0.01
Selected features: 33
Best dev MAE: 4.955494157406549
Test metrics:
{'mae': 4.671701740553086, 'rmse': 5.8956533025229785, 'r2': 0.15422467359768166, 'spearman_coefficient': 0.4762968130752058, 'spearman_p_value': 0.0008198458548478274}


,feature_index,feature,ngram_size,feature_type,contains_annotation,idf,coefficient,absolute_coefficient,coefficient_direction,alpha,importance_rank
0,3847,i'm alright,2,bigram_word_word,False,3.852631,50.085648,50.085648,positive,0.01,1
1,3165,guilty,1,unigram_word,False,2.508897,-47.712813,47.712813,negative,0.01,2
2,5118,medication,1,unigram_word,False,3.246496,44.102255,44.102255,positive,0.01,3
3,5793,of em,2,bigram_word_word,False,3.159484,39.784190,39.784190,positive,0.01,4
4,6315,people that,2,bigram_word_word,False,2.277095,34.778688,34.778688,positive,0.01,5
5,6749,relationship,1,unigram_word,False,2.089043,-33.238338,33.238338,negative,0.01,6
6,8057,therapist,1,unigram_word,False,2.936341,31.686026,31.686026,positive,0.01,7
7,8446,to handle,2,bigram_word_word,False,3.698481,31.457298,31.457298,positive,0.01,8
8,9548,whether,1,unigram_word,False,2.648659,-27.092205,27.092205,negative,0.01,9
9,757,annotation_sniff,1,unigram_annotation,True,3.447166,25.066426,25.066426,positive,0.01,10


### Experiment 4B: Binary depression L1 logistic search

In [106]:
experiment4_classification_results = []

experiment4_best_logistic_model = None

experiment4_best_C = None

experiment4_best_balanced_accuracy = -np.inf


for C in CLASSIFICATION_C_VALUES:

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=C,
        class_weight=LOGISTIC_CLASS_WEIGHT,
        max_iter=LOGISTIC_MAX_ITER,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_train_unigram_bigram_l1,
        y_train_binary,
    )

    coefficients = (
        model.coef_[0]
    )

    selected_count = int(
        np.sum(
            coefficients != 0
        )
    )

    # --------------------------------------------------------
    # TRAIN PREDICTIONS
    # --------------------------------------------------------

    train_predictions = (
        model.predict(
            X_train_unigram_bigram_l1
        )
    )

    train_probabilities = (
        model.predict_proba(
            X_train_unigram_bigram_l1
        )[:, 1]
    )

    # --------------------------------------------------------
    # DEV PREDICTIONS
    # --------------------------------------------------------

    dev_predictions = (
        model.predict(
            X_dev_unigram_bigram_l1
        )
    )

    dev_probabilities = (
        model.predict_proba(
            X_dev_unigram_bigram_l1
        )[:, 1]
    )

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    train_metrics = (
        calculate_classification_metrics(
            y_train_binary,
            train_predictions,
            train_probabilities,
        )
    )

    dev_metrics = (
        calculate_classification_metrics(
            y_dev_binary,
            dev_predictions,
            dev_probabilities,
        )
    )

    allowed = (
        feature_count_allowed(
            selected_count
        )
    )

    experiment4_classification_results.append(
        {
            "C":
                C,

            "selected_features":
                selected_count,

            "feature_count_allowed":
                allowed,

            # ---------------- TRAIN ----------------

            "train_balanced_accuracy":
                train_metrics[
                    "balanced_accuracy"
                ],

            "train_accuracy":
                train_metrics[
                    "accuracy"
                ],

            "train_f1":
                train_metrics[
                    "f1"
                ],

            "train_precision":
                train_metrics[
                    "precision"
                ],

            "train_recall_sensitivity":
                train_metrics[
                    "recall_sensitivity"
                ],

            "train_specificity":
                train_metrics[
                    "specificity"
                ],

            "train_roc_auc":
                train_metrics[
                    "roc_auc"
                ],

            # ---------------- DEV ----------------

            "dev_balanced_accuracy":
                dev_metrics[
                    "balanced_accuracy"
                ],

            "dev_accuracy":
                dev_metrics[
                    "accuracy"
                ],

            "dev_f1":
                dev_metrics[
                    "f1"
                ],

            "dev_precision":
                dev_metrics[
                    "precision"
                ],

            "dev_recall_sensitivity":
                dev_metrics[
                    "recall_sensitivity"
                ],

            "dev_specificity":
                dev_metrics[
                    "specificity"
                ],

            "dev_roc_auc":
                dev_metrics[
                    "roc_auc"
                ],
        }
    )

    # --------------------------------------------------------
    # SELECT BEST C USING DEV BALANCED ACCURACY
    # --------------------------------------------------------

    if (
        allowed
        and selected_count > 0
        and dev_metrics[
            "balanced_accuracy"
        ] > experiment4_best_balanced_accuracy
    ):

        experiment4_best_balanced_accuracy = (
            dev_metrics[
                "balanced_accuracy"
            ]
        )

        experiment4_best_C = C

        experiment4_best_logistic_model = (
            model
        )


experiment4_classification_search_df = (
    pd.DataFrame(
        experiment4_classification_results
    )
)

experiment4_classification_search_df

/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

,C,selected_features,feature_count_allowed,train_balanced_accuracy,train_accuracy,train_f1,train_precision,train_recall_sensitivity,train_specificity,train_roc_auc,dev_balanced_accuracy,dev_accuracy,dev_f1,dev_precision,dev_recall_sensitivity,dev_specificity,dev_roc_auc
0,0.0010,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
1,0.0025,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
2,0.0050,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
3,0.0100,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
4,0.0250,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
5,0.0500,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
6,0.1000,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
7,0.2500,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
8,0.5000,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000
9,1.0000,0,True,0.500000,0.718447,0.000,0.000000,0.000000,1.000000,0.500000,0.500000,0.65625,0.000000,0.000000,0.000000,1.000000,0.500000


In [107]:
# Save and evaluate Experiment 4B

if experiment4_best_logistic_model is None:
    raise RuntimeError(
        "No valid Experiment 4 "
        "logistic regression model found."
    )


EXPERIMENT_4_BINARY_DIR = (
    EXPERIMENT_4_DIR
    / "depression_binary"
)

EXPERIMENT_4_BINARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# TEST SET EVALUATION
# ============================================================

test_predictions = (
    experiment4_best_logistic_model.predict(
        X_test_unigram_bigram_l1
    )
)

test_probabilities = (
    experiment4_best_logistic_model.predict_proba(
        X_test_unigram_bigram_l1
    )[:, 1]
)

experiment4_binary_test_metrics = (
    calculate_classification_metrics(
        y_test_binary,
        test_predictions,
        test_probabilities,
    )
)


print(
    "Best C:",
    experiment4_best_C
)

print(
    "Selected features:",
    np.sum(
        experiment4_best_logistic_model.coef_[0]
        != 0
    )
)

print(
    "Best dev balanced accuracy:",
    experiment4_best_balanced_accuracy
)

print()

print(
    "Test metrics:"
)

print(
    experiment4_binary_test_metrics
)


# ============================================================
# SAVE REGULARIZATION SEARCH
# ============================================================

experiment4_classification_search_df.to_csv(
    EXPERIMENT_4_BINARY_DIR
    / "regularization_search.csv",
    index=False,
)


# ============================================================
# SAVE BEST MODEL
# ============================================================

joblib.dump(
    experiment4_best_logistic_model,
    EXPERIMENT_4_BINARY_DIR
    / "l1_logistic_model.joblib",
)


# ============================================================
# BUILD SELECTED FEATURE METADATA
# ============================================================

experiment4_selected_binary_features = (
    build_selected_features_dataframe(
        vocabulary_df=
            vocabulary_unigram_bigram_l1,

        coefficients=
            experiment4_best_logistic_model.coef_[0],

        parameter_name=
            "C",

        parameter_value=
            experiment4_best_C,
    )
)


experiment4_selected_binary_features.to_csv(
    EXPERIMENT_4_BINARY_DIR
    / "selected_features.csv",
    index=False,
)


# ============================================================
# CREATE REDUCED TF-IDF MATRICES
# ============================================================

(
    X_train_exp4_binary_selected,
    X_dev_exp4_binary_selected,
    X_test_exp4_binary_selected,
    exp4_binary_selected_indices,
) = get_selected_matrices(
    X_train_unigram_bigram_l1,
    X_dev_unigram_bigram_l1,
    X_test_unigram_bigram_l1,
    experiment4_best_logistic_model.coef_[0],
)


# ============================================================
# SAVE REDUCED MATRICES
# ============================================================

save_npz(
    EXPERIMENT_4_BINARY_DIR
    / "train_selected.npz",
    X_train_exp4_binary_selected,
)

save_npz(
    EXPERIMENT_4_BINARY_DIR
    / "dev_selected.npz",
    X_dev_exp4_binary_selected,
)

save_npz(
    EXPERIMENT_4_BINARY_DIR
    / "test_selected.npz",
    X_test_exp4_binary_selected,
)


# ============================================================
# SAVE ORIGINAL FEATURE INDICES
# ============================================================

np.save(
    EXPERIMENT_4_BINARY_DIR
    / "selected_feature_indices.npy",
    exp4_binary_selected_indices,
)


experiment4_selected_binary_features.head(30)

Best C: 5.0
Selected features: 8
Best dev balanced accuracy: 0.6493506493506493

Test metrics:
{'balanced_accuracy': 0.4642857142857143, 'accuracy': 0.4782608695652174, 'f1': 0.3333333333333333, 'precision': 0.2727272727272727, 'recall_sensitivity': 0.42857142857142855, 'specificity': 0.5, 'roc_auc': 0.4575892857142857}


,feature_index,feature,ngram_size,feature_type,contains_annotation,idf,coefficient,absolute_coefficient,coefficient_direction,C,importance_rank
0,2683,for me,2,bigram_word_word,False,1.732368,12.671549,12.671549,positive,5.0,1
1,8057,therapist,1,unigram_word,False,2.936341,9.185977,9.185977,positive,5.0,2
2,85,a lot,2,bigram_word_word,False,1.111791,3.297354,3.297354,positive,5.0,3
3,7047,she's,1,unigram_word,False,1.860201,2.322265,2.322265,positive,5.0,4
4,5793,of em,2,bigram_word_word,False,3.159484,1.618604,1.618604,positive,5.0,5
5,503,and i've,2,bigram_word_word,False,2.754019,1.551344,1.551344,positive,5.0,6
6,6487,probably a,2,bigram_word_word,False,3.246496,0.328287,0.328287,positive,5.0,7
7,7328,son,1,unigram_word,False,2.811178,0.299179,0.299179,positive,5.0,8


### Inspect annotation features selected for PHQ-8

In [108]:
experiment4_phq8_annotation_features = (
    experiment4_selected_phq8_features.loc[
        experiment4_selected_phq8_features[
            "contains_annotation"
        ]
    ]
    .copy()
)

print(
    "Selected PHQ-8 features containing annotations:",
    len(
        experiment4_phq8_annotation_features
    )
)

experiment4_phq8_annotation_features[
    [
        "feature",
        "feature_type",
        "idf",
        "coefficient",
        "absolute_coefficient",
        "coefficient_direction",
        "importance_rank",
    ]
].head(30)

Selected PHQ-8 features containing annotations: 1


,feature,feature_type,idf,coefficient,absolute_coefficient,coefficient_direction,importance_rank
9,annotation_sniff,unigram_annotation,3.447166,25.066426,25.066426,positive,10


### Inspect annotation features selected for binary depression

In [109]:
experiment4_binary_annotation_features = (
    experiment4_selected_binary_features.loc[
        experiment4_selected_binary_features[
            "contains_annotation"
        ]
    ]
    .copy()
)

print(
    "Selected binary features containing annotations:",
    len(
        experiment4_binary_annotation_features
    )
)

experiment4_binary_annotation_features[
    [
        "feature",
        "feature_type",
        "idf",
        "coefficient",
        "absolute_coefficient",
        "coefficient_direction",
        "importance_rank",
    ]
].head(30)

Selected binary features containing annotations: 0


,feature,feature_type,idf,coefficient,absolute_coefficient,coefficient_direction,importance_rank


### Compare PHQ-8 and binary selected features

In [110]:
phq8_feature_set = set(
    experiment4_selected_phq8_features[
        "feature"
    ]
)

binary_feature_set = set(
    experiment4_selected_binary_features[
        "feature"
    ]
)


shared_features = sorted(
    phq8_feature_set
    & binary_feature_set
)

phq8_only_features = sorted(
    phq8_feature_set
    - binary_feature_set
)

binary_only_features = sorted(
    binary_feature_set
    - phq8_feature_set
)


print(
    "PHQ-8 selected:",
    len(phq8_feature_set)
)

print(
    "Binary selected:",
    len(binary_feature_set)
)

print(
    "Shared:",
    len(shared_features)
)

print(
    "PHQ-8 only:",
    len(phq8_only_features)
)

print(
    "Binary only:",
    len(binary_only_features)
)

PHQ-8 selected: 33
Binary selected: 8
Shared: 4
PHQ-8 only: 29
Binary only: 4


In [111]:
shared_features_df = (
    experiment4_selected_phq8_features[
        [
            "feature",
            "feature_type",
            "contains_annotation",
            "coefficient",
            "importance_rank",
        ]
    ]
    .rename(
        columns={
            "coefficient":
                "phq8_coefficient",

            "importance_rank":
                "phq8_rank",
        }
    )
    .merge(
        experiment4_selected_binary_features[
            [
                "feature",
                "coefficient",
                "importance_rank",
            ]
        ].rename(
            columns={
                "coefficient":
                    "binary_coefficient",

                "importance_rank":
                    "binary_rank",
            }
        ),
        on="feature",
        how="inner",
    )
)

shared_features_df = (
    shared_features_df
    .sort_values(
        "phq8_rank"
    )
    .reset_index(drop=True)
)

shared_features_df.head(30)

,feature,feature_type,contains_annotation,phq8_coefficient,phq8_rank,binary_coefficient,binary_rank
0,of em,bigram_word_word,False,39.784190,4,1.618604,5
1,therapist,unigram_word,False,31.686026,7,9.185977,2
2,son,unigram_word,False,12.547490,18,0.299179,8
3,she's,unigram_word,False,6.662083,25,2.322265,4


In [112]:
shared_features_df.to_csv(
    EXPERIMENT_4_DIR
    / "shared_selected_features.csv",
    index=False,
)

### Inspect unigram vs bigram selection

In [113]:
# PHQ-8

print(
    experiment4_selected_phq8_features[
        "feature_type"
    ]
    .value_counts()
)

feature_type
unigram_word          17
bigram_word_word      15
unigram_annotation     1
Name: count, dtype: int64


In [114]:
# Binary depression

print(
    experiment4_selected_binary_features[
        "feature_type"
    ]
    .value_counts()
)

feature_type
bigram_word_word    5
unigram_word        3
Name: count, dtype: int64


### Inspect strongest positive and negative PHQ-8 features

In [115]:
# Positive coefficients

phq8_positive_features = (
    experiment4_selected_phq8_features.loc[
        experiment4_selected_phq8_features[
            "coefficient"
        ] > 0
    ]
    .sort_values(
        "coefficient",
        ascending=False,
    )
)

phq8_positive_features[
    [
        "feature",
        "feature_type",
        "coefficient",
        "idf",
        "importance_rank",
    ]
].head(30)

,feature,feature_type,coefficient,idf,importance_rank
0,i'm alright,bigram_word_word,50.085648,3.852631,1
2,medication,unigram_word,44.102255,3.246496,3
3,of em,bigram_word_word,39.784190,3.159484,4
4,people that,bigram_word_word,34.778688,2.277095,5
6,therapist,unigram_word,31.686026,2.936341,7
7,to handle,bigram_word_word,31.457298,3.698481,8
9,annotation_sniff,unigram_annotation,25.066426,3.447166,10
11,lot more,bigram_word_word,18.815481,2.699952,12
13,to my,bigram_word_word,15.870703,2.006805,14
14,depressed,unigram_word,14.893690,2.277095,15


In [116]:
# Negative coefficients

phq8_negative_features = (
    experiment4_selected_phq8_features.loc[
        experiment4_selected_phq8_features[
            "coefficient"
        ] < 0
    ]
    .sort_values(
        "coefficient",
        ascending=True,
    )
)

phq8_negative_features[
    [
        "feature",
        "feature_type",
        "coefficient",
        "idf",
        "importance_rank",
    ]
].head(30)

,feature,feature_type,coefficient,idf,importance_rank
1,guilty,unigram_word,-47.712813,2.508897,2
5,relationship,unigram_word,-33.238338,2.089043,6
8,whether,unigram_word,-27.092205,2.648659,9
10,party,unigram_word,-23.395330,3.079442,11
12,to make,bigram_word_word,-16.395663,1.980829,13
15,cultures,unigram_word,-13.888587,2.936341,16
16,and you,bigram_word_word,-12.681797,1.773190,17
18,not really,bigram_word_word,-12.183104,1.693147,19
20,even,unigram_word,-11.289312,1.566853,21
22,hmm,unigram_word,-10.410216,1.860201,23


### Strongest binary classification features

In [117]:
# Features pushing toward depression = 1

binary_positive_features = (
    experiment4_selected_binary_features.loc[
        experiment4_selected_binary_features[
            "coefficient"
        ] > 0
    ]
    .sort_values(
        "coefficient",
        ascending=False,
    )
)

binary_positive_features[
    [
        "feature",
        "feature_type",
        "coefficient",
        "idf",
        "importance_rank",
    ]
].head(30)

,feature,feature_type,coefficient,idf,importance_rank
0,for me,bigram_word_word,12.671549,1.732368,1
1,therapist,unigram_word,9.185977,2.936341,2
2,a lot,bigram_word_word,3.297354,1.111791,3
3,she's,unigram_word,2.322265,1.860201,4
4,of em,bigram_word_word,1.618604,3.159484,5
5,and i've,bigram_word_word,1.551344,2.754019,6
6,probably a,bigram_word_word,0.328287,3.246496,7
7,son,unigram_word,0.299179,2.811178,8


In [118]:
# Features pushing toward depression = 0

binary_negative_features = (
    experiment4_selected_binary_features.loc[
        experiment4_selected_binary_features[
            "coefficient"
        ] < 0
    ]
    .sort_values(
        "coefficient",
        ascending=True,
    )
)

binary_negative_features[
    [
        "feature",
        "feature_type",
        "coefficient",
        "idf",
        "importance_rank",
    ]
].head(30)

,feature,feature_type,coefficient,idf,importance_rank
